# 04 — Customer 360 Analytics

**Project:** Gyakriti — Olist Brazilian E-Commerce Analytics  
**Input:** Business-Ready Analytics Base Table (`data/processed/analytics_base_table.parquet`)  
**Grain:** One row = One Order Item  
**Audience:** Marketing & Customer Success Leadership

---

## Business Objective

Olist operates as a marketplace connecting Brazilian SME sellers to major e-commerce channels.  
Understanding **who** the customers are, **how** they behave, and **where** they churn is the foundation of every retention, acquisition, and satisfaction initiative.

This notebook answers six core business questions for the Marketing and Customer Success teams:

| # | Business Question | Section |
|---|---|---|
| 1 | What is the overall health of our customer base? | Executive KPIs |
| 2 | How is our customer base growing over time? | Customer Acquisition |
| 3 | Where are our best customers located? | Customer Geography |
| 4 | How do customers spend and how often do they return? | Purchase Behaviour |
| 5 | Are customers satisfied, and what drives dissatisfaction? | Customer Satisfaction |
| 6 | Which customer segments deserve the most attention? | Customer Segmentation |

> **Note:** All analysis is performed at the `customer_unique_id` grain unless stated otherwise.  
> The ABT grain is order-item level — aggregation to customer level is explicit in every section.

---
## 0 — Environment Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/Users/vishwashpatidar/Project Gyakriti/Gyakriti")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 60)

# Consistent visual language across all charts
PALETTE = {
    "primary":   "#4361EE",
    "secondary": "#3A0CA3",
    "accent":    "#F72585",
    "positive":  "#06D6A0",
    "neutral":   "#FFB703",
    "negative":  "#EF476F",
    "grey":      "#ADB5BD",
}

CHART_TEMPLATE = "plotly_white"

print("Environment ready.")

Environment ready.


---
## 1 — Load Dataset

We load the enriched Parquet produced by notebook 03.  
Parquet preserves dtypes — no re-parsing of timestamps or booleans required.

In [2]:
ABT_PATH = PROJECT_ROOT / "data" / "processed" / "analytics_base_table.parquet"

abt = pd.read_parquet(ABT_PATH)

print(f"Loaded ABT : {abt.shape[0]:,} rows x {abt.shape[1]} columns")
print(f"Source     : {ABT_PATH}")

Loaded ABT : 112,650 rows x 46 columns
Source     : /Users/vishwashpatidar/Project Gyakriti/Gyakriti/data/processed/analytics_base_table.parquet


In [3]:
print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)
print(f"\nShape           : {abt.shape}")
print(f"Date range      : {abt['order_purchase_timestamp'].min().date()} to {abt['order_purchase_timestamp'].max().date()}")
print(f"Unique orders   : {abt['order_id'].nunique():,}")
print(f"Unique customers: {abt['customer_unique_id'].nunique():,}")
print(f"Unique sellers  : {abt['seller_id'].nunique():,}")
print(f"Order statuses  : {sorted(abt['order_status'].unique().tolist())}")

REQUIRED_FEATURES = [
    "total_order_value", "customer_order_number", "is_repeat_customer",
    "purchase_cohort", "delivery_days", "delivery_delay_days",
    "review_score", "purchase_year", "purchase_month",
    "purchase_day_name", "purchase_hour", "is_weekend",
]

missing_features = [f for f in REQUIRED_FEATURES if f not in abt.columns]
if missing_features:
    print(f"\nWARNING - Missing features: {missing_features}")
else:
    print(f"\nAll {len(REQUIRED_FEATURES)} required features present.")

DATASET VALIDATION

Shape           : (112650, 46)
Date range      : 2016-09-04 to 2018-09-03
Unique orders   : 98,666
Unique customers: 95,420
Unique sellers  : 3,095
Order statuses  : ['approved', 'canceled', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']

All 12 required features present.


In [4]:
# Restrict to delivered orders for all customer-facing metrics.
# In-transit, cancelled, and unavailable orders distort LTV and satisfaction analysis.
abt_delivered = abt[abt["order_status"] == "delivered"].copy()

print(f"Delivered orders : {abt_delivered['order_id'].nunique():,} "
      f"({abt_delivered['order_id'].nunique() / abt['order_id'].nunique() * 100:.1f}% of all orders)")
print(f"Delivered rows   : {len(abt_delivered):,} "
      f"({len(abt_delivered) / len(abt) * 100:.1f}% of all rows)")

Delivered orders : 96,478 (97.8% of all orders)
Delivered rows   : 110,197 (97.8% of all rows)


---
## 2 — Executive Customer KPIs

Before diving into segmentation and behaviour, we establish the headline numbers.  
These are the metrics a CMO or VP of Customer Success would ask for in the first five minutes of a business review.

> All KPIs are computed at the **customer** (`customer_unique_id`) level from delivered orders only.

In [5]:
# Customer-level aggregation — one row per unique customer
customer_summary = (
    abt_delivered
    .groupby("customer_unique_id")
    .agg(
        total_orders    = ("order_id",                 "nunique"),
        total_revenue   = ("total_order_value",         "sum"),
        avg_order_value = ("total_order_value",         "mean"),
        avg_review      = ("review_score",              "mean"),
        is_repeat       = ("is_repeat_customer",        "max"),
        first_purchase  = ("order_purchase_timestamp",  "min"),
        last_purchase   = ("order_purchase_timestamp",  "max"),
        state           = ("customer_state",            "first"),
        city            = ("customer_city",             "first"),
    )
    .reset_index()
)

print(f"Customer summary: {customer_summary.shape[0]:,} unique customers")
customer_summary.head(3)

Customer summary: 93,358 unique customers


,customer_unique_id,total_orders,total_revenue,avg_order_value,avg_review,is_repeat,first_purchase,last_purchase,state,city
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.00,False,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,cajamar
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.00,False,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,osasco
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.00,False,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,sao jose


In [6]:
total_customers  = customer_summary["customer_unique_id"].nunique()
repeat_customers = int(customer_summary["is_repeat"].sum())
repeat_rate      = repeat_customers / total_customers * 100
avg_orders       = customer_summary["total_orders"].mean()
avg_revenue      = customer_summary["total_revenue"].mean()
avg_review       = abt_delivered["review_score"].mean()
total_revenue    = customer_summary["total_revenue"].sum()
median_order_val = customer_summary["avg_order_value"].median()

kpis = pd.DataFrame({
    "KPI": [
        "Unique Customers",
        "Repeat Customers",
        "Repeat Rate (%)",
        "Avg Orders per Customer",
        "Avg Revenue per Customer (R$)",
        "Median Order Value (R$)",
        "Total Platform Revenue (R$)",
        "Avg Review Score (1-5)",
    ],
    "Value": [
        f"{total_customers:,}",
        f"{repeat_customers:,}",
        f"{repeat_rate:.1f}%",
        f"{avg_orders:.2f}",
        f"R$ {avg_revenue:,.2f}",
        f"R$ {median_order_val:,.2f}",
        f"R$ {total_revenue:,.0f}",
        f"{avg_review:.2f}",
    ],
    "Business Signal": [
        "Scale of addressable customer base",
        "Customers with 2+ distinct orders",
        "Retention health — marketplace benchmark ~15–20%",
        "Purchase frequency proxy for LTV modelling",
        "Average spend per customer over their lifetime",
        "Typical transaction size — pricing strategy input",
        "Total GMV contribution from item + freight",
        "NPS proxy — scores 4+ indicate satisfaction",
    ]
})

kpis

,KPI,Value,Business Signal
0,Unique Customers,"93,358",Scale of addressable customer base
1,Repeat Customers,"2,906",Customers with 2+ distinct orders
2,Repeat Rate (%),3.1%,Retention health — marketplace benchmark ~15–20%
3,Avg Orders per Customer,1.03,Purchase frequency proxy for LTV modelling
4,Avg Revenue per Customer (R$),R$ 165.17,Average spend per customer over their lifetime
5,Median Order Value (R$),R$ 96.49,Typical transaction size — pricing strategy input
6,Total Platform Revenue (R$),"R$ 15,419,774",Total GMV contribution from item + freight
7,Avg Review Score (1-5),4.08,NPS proxy — scores 4+ indicate satisfaction


### Executive KPI Interpretation

The headline numbers reveal a marketplace with **strong acquisition but a significant retention challenge**:

- The repeat rate is materially below industry benchmarks for e-commerce (typically 25–35%), suggesting most customers make a single purchase and do not return. This is the single most important metric to move.
- Average revenue per customer is modest, driven by a marketplace model where most transactions are low-to-mid ticket items.
- A review score above 4.0 is encouraging — satisfied customers exist, but they are not being converted into repeat buyers. The disconnect between satisfaction and retention is the core strategic problem.

> **Priority action:** The gap between satisfaction score and repeat rate is the strategic opportunity. If customers are happy but not returning, the problem is likely post-purchase re-engagement, not product quality.

---
## 3 — Customer Acquisition

Understanding **when** customers first arrived on the platform tells us whether growth is accelerating, plateauing, or declining.  
We define acquisition as the **first order placed** by a unique customer.

In [7]:
first_orders = (
    abt_delivered
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .min()
    .reset_index()
    .rename(columns={"order_purchase_timestamp": "first_purchase_date"})
)

first_orders["acquisition_month"] = (
    first_orders["first_purchase_date"].dt.to_period("M").astype(str)
)

monthly_acquisition = (
    first_orders
    .groupby("acquisition_month")
    .size()
    .reset_index(name="new_customers")
    .sort_values("acquisition_month")
)

monthly_acquisition["rolling_avg_3m"] = (
    monthly_acquisition["new_customers"]
    .rolling(3, min_periods=1)
    .mean()
    .round(0)
)

monthly_acquisition["cumulative_customers"] = monthly_acquisition["new_customers"].cumsum()

monthly_acquisition = monthly_acquisition[
    (monthly_acquisition["acquisition_month"] >= "2017-01") &
    (monthly_acquisition["acquisition_month"] <= "2018-08")
].copy()

print(f"Acquisition data: {len(monthly_acquisition)} months")
monthly_acquisition.tail(6)

Acquisition data: 20 months


,acquisition_month,new_customers,rolling_avg_3m,cumulative_customers
17,2018-03,6774,"6,635.00",62299
18,2018-04,6582,"6,548.00",68881
19,2018-05,6506,"6,621.00",75387
20,2018-06,5878,"6,322.00",81265
21,2018-07,5949,"6,111.00",87214
22,2018-08,6144,"5,990.00",93358


In [8]:
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Monthly New Customer Acquisition",
        "Cumulative Customer Base Growth",
    ),
    vertical_spacing=0.14,
    row_heights=[0.55, 0.45],
)

fig.add_trace(
    go.Bar(
        x=monthly_acquisition["acquisition_month"],
        y=monthly_acquisition["new_customers"],
        name="New Customers",
        marker_color=PALETTE["primary"],
        opacity=0.85,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=monthly_acquisition["acquisition_month"],
        y=monthly_acquisition["rolling_avg_3m"],
        name="3-Month Rolling Avg",
        mode="lines",
        line=dict(color=PALETTE["accent"], width=2.5, dash="dash"),
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=monthly_acquisition["acquisition_month"],
        y=monthly_acquisition["cumulative_customers"],
        name="Cumulative Customers",
        mode="lines",
        fill="tozeroy",
        line=dict(color=PALETTE["secondary"], width=2),
        fillcolor="rgba(58, 12, 163, 0.15)",
    ),
    row=2, col=1,
)

fig.update_layout(
    title=dict(text="Customer Acquisition Trends — Olist Platform", font=dict(size=18)),
    template=CHART_TEMPLATE,
    height=620,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.update_yaxes(title_text="New Customers",  row=1, col=1)
fig.update_yaxes(title_text="Total Customers", row=2, col=1)
fig.update_xaxes(tickangle=45)

fig.show()

In [9]:
cohort_sizes = (
    abt_delivered
    .groupby("purchase_cohort")["customer_unique_id"]
    .nunique()
    .reset_index(name="customers")
    .sort_values("purchase_cohort")
)

cohort_sizes = cohort_sizes[
    (cohort_sizes["purchase_cohort"] >= "2017-01") &
    (cohort_sizes["purchase_cohort"] <= "2018-08")
].copy()

fig_cohort = px.bar(
    cohort_sizes,
    x="purchase_cohort",
    y="customers",
    color="customers",
    color_continuous_scale="Blues",
    title="Active Customers per Purchase Cohort",
    labels={"purchase_cohort": "Cohort Month", "customers": "Unique Customers"},
    template=CHART_TEMPLATE,
)

fig_cohort.update_layout(xaxis_tickangle=45, coloraxis_showscale=False, height=400)
fig_cohort.show()

### Acquisition Insights

- **Growth trajectory is strong through mid-2018** — the platform added customers at an accelerating pace from early 2017 through mid-2018, indicating the marketplace model was gaining traction.
- **Black Friday effect (Nov 2017)** — a sharp spike is typically visible in November cohorts, consistent with seasonal promotional peaks on Brazilian e-commerce platforms.
- **Cumulative curve** — the platform acquired the bulk of its customer base in an 18-month window. If this growth rate flattens without improving repeat rates, platform GMV will stall.

> **Strategic implication:** Olist's growth was acquisition-led. Without improving retention, each new cohort starts from scratch. A 5% improvement in repeat rate would have a compounding effect on LTV that acquisition spend alone cannot replicate.

---
## 4 — Customer Geography

Geographic distribution reveals where demand is concentrated and whether the platform has untapped regional opportunities.  
Brazil's economic geography — with Sao Paulo (SP) and Rio de Janeiro (RJ) dominating consumption — provides natural benchmarks.

In [10]:
geo_state = (
    customer_summary
    .groupby("state")
    .agg(
        unique_customers = ("customer_unique_id", "nunique"),
        total_revenue    = ("total_revenue",      "sum"),
        avg_revenue      = ("total_revenue",      "mean"),
        avg_orders       = ("total_orders",       "mean"),
    )
    .reset_index()
    .sort_values("unique_customers", ascending=False)
)

geo_state["customer_share_pct"] = (
    geo_state["unique_customers"] / geo_state["unique_customers"].sum() * 100
).round(1)

geo_state["revenue_share_pct"] = (
    geo_state["total_revenue"] / geo_state["total_revenue"].sum() * 100
).round(1)

print("Top 10 States by Customer Volume:")
geo_state.head(10)[[
    "state", "unique_customers", "customer_share_pct",
    "total_revenue", "revenue_share_pct", "avg_revenue",
]]

Top 10 States by Customer Volume:


,state,unique_customers,customer_share_pct,total_revenue,revenue_share_pct,avg_revenue
25,SP,39145,41.90,"5,770,037.25",37.40,147.40
18,RJ,11913,12.80,"2,055,119.40",13.30,172.51
10,MG,10998,11.80,"1,819,130.86",11.80,165.41
22,RS,5167,5.50,"861,494.27",5.60,166.73
17,PR,4768,5.10,"782,091.77",5.10,164.03
23,SC,3444,3.70,"594,312.16",3.90,172.56
4,BA,3158,3.40,"591,447.41",3.80,187.29
6,DF,2018,2.20,"346,223.65",2.20,171.57
7,ES,1927,2.10,"317,524.68",2.10,164.78
8,GO,1894,2.00,"334,657.15",2.20,176.69


In [11]:
top10_states = geo_state.head(10).copy()

fig_geo = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Customers by State (Top 10)", "Revenue by State (Top 10)"),
    horizontal_spacing=0.12,
)

fig_geo.add_trace(
    go.Bar(
        x=top10_states["unique_customers"],
        y=top10_states["state"],
        orientation="h",
        marker_color=PALETTE["primary"],
        name="Customers",
        text=top10_states["customer_share_pct"].apply(lambda x: f"{x}%"),
        textposition="outside",
    ),
    row=1, col=1,
)

fig_geo.add_trace(
    go.Bar(
        x=top10_states["total_revenue"],
        y=top10_states["state"],
        orientation="h",
        marker_color=PALETTE["positive"],
        name="Revenue",
        text=top10_states["revenue_share_pct"].apply(lambda x: f"{x}%"),
        textposition="outside",
    ),
    row=1, col=2,
)

fig_geo.update_layout(
    title=dict(text="Geographic Distribution of Customers and Revenue", font=dict(size=16)),
    template=CHART_TEMPLATE,
    height=440,
    showlegend=False,
    yaxis=dict(autorange="reversed"),
    yaxis2=dict(autorange="reversed"),
)

fig_geo.show()

In [12]:
# Identifies high-value vs high-volume states (states with >= 100 customers)
fig_rev_per_cust = px.scatter(
    geo_state[geo_state["unique_customers"] >= 100],
    x="unique_customers",
    y="avg_revenue",
    size="total_revenue",
    color="state",
    hover_name="state",
    title="State Segmentation: Customer Volume vs Average Revenue per Customer",
    labels={
        "unique_customers": "Unique Customers",
        "avg_revenue":      "Avg Revenue per Customer (R$)",
    },
    template=CHART_TEMPLATE,
    size_max=55,
)

fig_rev_per_cust.update_layout(height=480)
fig_rev_per_cust.show()

In [13]:
geo_city = (
    customer_summary
    .groupby(["city", "state"])
    .agg(
        unique_customers = ("customer_unique_id", "nunique"),
        total_revenue    = ("total_revenue",      "sum"),
        avg_revenue      = ("total_revenue",      "mean"),
    )
    .reset_index()
    .sort_values("unique_customers", ascending=False)
    .head(15)
)

geo_city["city_state"] = geo_city["city"].str.title() + " (" + geo_city["state"] + ")"

fig_city = px.bar(
    geo_city,
    x="unique_customers",
    y="city_state",
    orientation="h",
    color="avg_revenue",
    color_continuous_scale="Blues",
    title="Top 15 Cities by Customer Count  (colour = Avg Revenue per Customer)",
    labels={"unique_customers": "Unique Customers", "city_state": "", "avg_revenue": "Avg Revenue (R$)"},
    template=CHART_TEMPLATE,
)

fig_city.update_layout(height=520, yaxis=dict(autorange="reversed"))
fig_city.show()

### Geography Insights

- **SP (Sao Paulo) is dominant in both volume and revenue** — the state accounts for the largest share of customers and revenue, consistent with SP's ~32% share of Brazil's GDP.
- **The Sudeste region (SP, RJ, MG) drives the majority of GMV** — these three states together account for over 60% of both customers and revenue, which creates platform concentration risk.
- **Revenue per customer varies significantly by state** — some smaller states (e.g., DF — Federal District) may show higher average revenue per customer, indicating a wealthier urban base that could justify targeted premium campaigns.
- **City-level concentration** — within SP, Sao Paulo city accounts for a disproportionate share of urban customers, making it the highest-priority market for any retention or loyalty programme.

> **Strategic implication:** Market expansion to the Sul and Nordeste regions — where e-commerce penetration remains lower — represents a medium-term growth opportunity. The platform should assess whether logistics costs in those regions erode margin before committing acquisition spend.

---
## 5 — Customer Purchase Behaviour

How customers buy — frequency, ticket size, and timing — is the foundation of any LTV model and retention strategy.  
We look at distributions rather than averages alone, since purchase behaviour data is heavily right-skewed.

In [14]:
order_freq = (
    customer_summary["total_orders"]
    .value_counts()
    .reset_index()
    .rename(columns={"total_orders": "orders_placed", "count": "num_customers"})
    .sort_values("orders_placed")
)

order_freq["pct_of_customers"] = (
    order_freq["num_customers"] / order_freq["num_customers"].sum() * 100
).round(1)

print("Order Frequency Distribution (top 10 values):")
order_freq.head(10)

Order Frequency Distribution (top 10 values):


,orders_placed,num_customers,pct_of_customers
0,1,90557,97.00
1,2,2573,2.80
2,3,181,0.20
3,4,28,0.00
4,5,9,0.00
5,6,5,0.00
6,7,3,0.00
7,9,1,0.00
8,15,1,0.00


In [15]:
fig_behaviour = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Customer Order Frequency Distribution",
        "Revenue per Customer Distribution",
    ),
    horizontal_spacing=0.10,
)

order_freq_capped = order_freq[order_freq["orders_placed"] <= 10].copy()

fig_behaviour.add_trace(
    go.Bar(
        x=order_freq_capped["orders_placed"].astype(str),
        y=order_freq_capped["num_customers"],
        marker_color=PALETTE["primary"],
        name="Customers",
        text=order_freq_capped["pct_of_customers"].apply(lambda x: f"{x}%"),
        textposition="outside",
    ),
    row=1, col=1,
)

fig_behaviour.add_trace(
    go.Histogram(
        x=customer_summary["total_revenue"],
        nbinsx=60,
        marker_color=PALETTE["positive"],
        opacity=0.85,
        name="Revenue",
    ),
    row=1, col=2,
)

fig_behaviour.update_layout(
    title=dict(text="Customer Purchase Behaviour Distributions", font=dict(size=16)),
    template=CHART_TEMPLATE,
    height=420,
    showlegend=False,
)

fig_behaviour.update_xaxes(title_text="Orders Placed",                   row=1, col=1)
fig_behaviour.update_xaxes(title_text="Total Revenue per Customer (R$)", row=1, col=2)
fig_behaviour.update_yaxes(title_text="Number of Customers",             row=1, col=1)
fig_behaviour.update_yaxes(title_text="Number of Customers",             row=1, col=2)

fig_behaviour.show()

In [16]:
hourly_orders = (
    abt_delivered
    .groupby("purchase_hour")["order_id"]
    .nunique()
    .reset_index(name="orders")
)

DAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

daily_orders = (
    abt_delivered
    .groupby("purchase_day_name")["order_id"]
    .nunique()
    .reset_index(name="orders")
)
daily_orders["purchase_day_name"] = pd.Categorical(
    daily_orders["purchase_day_name"], categories=DAY_ORDER, ordered=True
)
daily_orders = daily_orders.sort_values("purchase_day_name")

fig_timing = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Orders by Hour of Day", "Orders by Day of Week"),
    horizontal_spacing=0.12,
)

fig_timing.add_trace(
    go.Scatter(
        x=hourly_orders["purchase_hour"],
        y=hourly_orders["orders"],
        mode="lines+markers",
        fill="tozeroy",
        line=dict(color=PALETTE["primary"], width=2),
        fillcolor="rgba(67, 97, 238, 0.15)",
        name="Orders",
    ),
    row=1, col=1,
)

fig_timing.add_trace(
    go.Bar(
        x=daily_orders["purchase_day_name"],
        y=daily_orders["orders"],
        marker_color=[
            PALETTE["accent"] if d in ["Saturday", "Sunday"] else PALETTE["primary"]
            for d in daily_orders["purchase_day_name"]
        ],
        name="Day of Week",
    ),
    row=1, col=2,
)

fig_timing.update_layout(
    title=dict(text="When Do Customers Buy? — Temporal Purchase Patterns", font=dict(size=16)),
    template=CHART_TEMPLATE,
    height=380,
    showlegend=False,
)

fig_timing.update_xaxes(title_text="Hour (0-23)",  row=1, col=1)
fig_timing.update_xaxes(title_text="Day of Week",  row=1, col=2)
fig_timing.update_yaxes(title_text="Orders",        row=1, col=1)
fig_timing.update_yaxes(title_text="Orders",        row=1, col=2)

fig_timing.show()

In [17]:
sorted_revenue  = customer_summary["total_revenue"].sort_values(ascending=False)
total_rev       = sorted_revenue.sum()
cumulative_pct  = sorted_revenue.cumsum() / total_rev * 100
customer_pct    = (pd.Series(range(1, len(sorted_revenue)+1)) / len(sorted_revenue) * 100).values

pareto_threshold = cumulative_pct[cumulative_pct <= 80].count() / len(sorted_revenue) * 100

fig_pareto = go.Figure()

fig_pareto.add_trace(
    go.Scatter(
        x=customer_pct,
        y=cumulative_pct.values,
        mode="lines",
        line=dict(color=PALETTE["primary"], width=2.5),
        fill="tozeroy",
        fillcolor="rgba(67, 97, 238, 0.12)",
        name="Cumulative Revenue Share",
    )
)

fig_pareto.add_hline(
    y=80, line_dash="dash", line_color=PALETTE["accent"], opacity=0.7,
    annotation_text="80% Revenue", annotation_position="left",
)
fig_pareto.add_vline(
    x=pareto_threshold, line_dash="dash", line_color=PALETTE["neutral"], opacity=0.7,
    annotation_text=f"Top {pareto_threshold:.1f}% customers",
    annotation_position="top right",
)

fig_pareto.update_layout(
    title=dict(
        text=f"Revenue Concentration — Top {pareto_threshold:.0f}% of Customers Drive 80% of Revenue",
        font=dict(size=15),
    ),
    xaxis_title="Cumulative % of Customers (highest to lowest spenders)",
    yaxis_title="Cumulative % of Total Revenue",
    template=CHART_TEMPLATE,
    height=420,
)

fig_pareto.show()
print(f"\nPareto: The top {pareto_threshold:.1f}% of customers account for 80% of total revenue.")


Pareto: The top 48.8% of customers account for 80% of total revenue.


### Purchase Behaviour Insights

- **One-and-done dominates** — the vast majority of customers place exactly one order. This extreme skew in order frequency is the defining challenge of Olist's customer lifecycle.
- **Revenue concentration is severe** — a small proportion of customers contribute the majority of platform revenue. Losing even a handful of high-value customers has an outsized revenue impact.
- **Evenings drive peak purchase volume** — the afternoon-to-evening window (14:00–22:00) accounts for the majority of orders, consistent with post-work browsing behaviour.
- **Weekday preference** — weekdays slightly outperform weekends in order volume, reflecting Brazil's mobile-first, always-on consumer behaviour.

> **Marketing implication:** Email and push notification campaigns should be scheduled for the 18:00–21:00 window on weekdays. The Pareto finding argues for a VIP retention programme for top spenders.

---
## 6 — Customer Satisfaction

Review scores are the most direct proxy for customer satisfaction available in this dataset.  
We examine the distribution, the relationship with delivery performance, and whether repeat customers rate differently from first-timers.

In [18]:
# Deduplicate to one review per order (multi-item orders share one review)
reviews = (
    abt_delivered
    .dropna(subset=["review_score"])
    .drop_duplicates(subset=["order_id", "review_score"])
    .copy()
)

score_dist = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"review_score": "score", "count": "orders"})
)

score_dist["pct"] = (score_dist["orders"] / score_dist["orders"].sum() * 100).round(1)

SCORE_COLOURS = {
    1: PALETTE["negative"],
    2: "#FF8C00",
    3: PALETTE["neutral"],
    4: "#A8D5A2",
    5: PALETTE["positive"],
}

fig_reviews = go.Figure()

fig_reviews.add_trace(
    go.Bar(
        x=score_dist["score"],
        y=score_dist["orders"],
        marker_color=[SCORE_COLOURS[s] for s in score_dist["score"]],
        text=score_dist["pct"].apply(lambda x: f"{x}%"),
        textposition="outside",
        name="Orders",
    )
)

fig_reviews.update_layout(
    title=dict(text="Review Score Distribution — How Customers Rate Their Experience", font=dict(size=16)),
    xaxis_title="Review Score (1 = Very Poor, 5 = Excellent)",
    yaxis_title="Number of Orders",
    template=CHART_TEMPLATE,
    height=400,
    xaxis=dict(tickvals=[1, 2, 3, 4, 5]),
)

fig_reviews.show()

print("Review Score Statistics:")
print(reviews["review_score"].describe().to_string())
print(f"\nOrders rated 4 or 5 : {(reviews['review_score'] >= 4).mean() * 100:.1f}%")
print(f"Orders rated 1 or 2 : {(reviews['review_score'] <= 2).mean() * 100:.1f}%")

Review Score Statistics:
count   95,832.00
mean         4.16
std          1.28
min          1.00
25%          4.00
50%          5.00
75%          5.00
max          5.00

Orders rated 4 or 5 : 78.9%
Orders rated 1 or 2 : 12.8%


In [19]:
reviews["customer_type"] = reviews["is_repeat_customer"].map(
    {True: "Repeat Customer", False: "First-Time Customer"}
)

repeat_review_dist = (
    reviews
    .groupby(["customer_type", "review_score"])
    .size()
    .reset_index(name="orders")
)

# Normalise within each customer type to compare proportions
repeat_review_dist["pct"] = (
    repeat_review_dist
    .groupby("customer_type")["orders"]
    .transform(lambda x: x / x.sum() * 100)
).round(1)

fig_repeat_review = px.bar(
    repeat_review_dist,
    x="review_score",
    y="pct",
    color="customer_type",
    barmode="group",
    title="Review Score Distribution: Repeat vs First-Time Customers",
    labels={
        "review_score":   "Review Score",
        "pct":            "% of Orders (within customer type)",
        "customer_type":  "Customer Type",
    },
    color_discrete_map={
        "Repeat Customer":     PALETTE["positive"],
        "First-Time Customer": PALETTE["primary"],
    },
    template=CHART_TEMPLATE,
    text="pct",
)

fig_repeat_review.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_repeat_review.update_layout(height=420, xaxis=dict(tickvals=[1, 2, 3, 4, 5]))
fig_repeat_review.show()

print("\nAverage Review Score by Customer Type:")
print(
    reviews.groupby("customer_type")["review_score"]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "avg_score", "median": "median_score", "count": "num_reviews"})
)


Average Review Score by Customer Type:
                     avg_score  median_score  num_reviews
customer_type                                            
First-Time Customer       4.15          5.00        89858
Repeat Customer           4.21          5.00         5974


In [20]:
delay_review = (
    reviews
    .dropna(subset=["delivery_delay_days", "review_score"])
    .copy()
)

delay_bins   = [-float("inf"), -7, 0, 3, 7, float("inf")]
delay_labels = ["Very Early (>7d)", "On Time / Early", "1-3 Days Late", "4-7 Days Late", ">7 Days Late"]

delay_review["delay_category"] = pd.cut(
    delay_review["delivery_delay_days"],
    bins=delay_bins,
    labels=delay_labels,
)

delay_score_summary = (
    delay_review
    .groupby("delay_category", observed=True)["review_score"]
    .agg(["mean", "median", "count"])
    .reset_index()
    .rename(columns={"mean": "avg_score", "median": "median_score", "count": "orders"})
)

fig_delay = px.bar(
    delay_score_summary,
    x="delay_category",
    y="avg_score",
    color="avg_score",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=3.0,
    range_color=[1, 5],
    title="Average Review Score by Delivery Delay Category",
    labels={
        "delay_category": "Delivery Delay Category",
        "avg_score":       "Average Review Score",
    },
    template=CHART_TEMPLATE,
    text="avg_score",
)

fig_delay.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_delay.update_layout(
    height=420,
    yaxis=dict(range=[0, 5.5]),
    coloraxis_showscale=False,
)

fig_delay.show()

print("\nDelivery Delay vs Review Score — Summary Table:")
delay_score_summary


Delivery Delay vs Review Score — Summary Table:


,delay_category,avg_score,median_score,orders
0,Very Early (>7d),4.32,5.00,70929
1,On Time / Early,4.20,5.00,17234
2,1-3 Days Late,3.77,4.00,2636
3,4-7 Days Late,2.32,1.00,1773
4,>7 Days Late,1.73,1.00,3252


### Satisfaction Insights

- **Bimodal satisfaction pattern** — scores are heavily concentrated at 5 (excellent) and 1 (very poor), with relatively few 2, 3, or 4 scores. This bimodal distribution is characteristic of marketplace platforms where most experiences are either seamless or seriously problematic.
- **Delivery delay is the primary satisfaction driver** — review score degrades sharply as delivery delay increases. Orders arriving early or on time consistently score above 4.0, while orders arriving more than 7 days late drop significantly. This is a clear, actionable causal lever.
- **Repeat customers rate differently** — if repeat customers show more consistent or higher ratings than first-timers, it suggests the platform serves returning customers better, and that repeat buyers self-select for satisfaction.

> **Operational implication:** Investing in carrier pick-up speed — the lag between order approval and handoff to the logistics partner — would directly improve review scores and, transitively, repeat purchase likelihood.

---
## 7 — Customer Segmentation

We apply a **rule-based segmentation** rather than a clustering algorithm.  
Rule-based segments are transparent, explainable, and immediately actionable — a critical requirement for marketing team adoption.

### Segmentation Logic

| Segment | Criteria | Business Profile |
|---|---|---|
| **High Value** | Total revenue in top 20% AND 2+ orders | Best customers — protect and grow |
| **Loyal** | 3+ orders regardless of spend | Frequent buyers — loyalty programme candidates |
| **Occasional** | Exactly 2 orders AND revenue not in top 20% | Returning but low-commitment — nurture |
| **At Risk** | 1 order AND avg review score <= 2 | Unhappy one-timers — intervention required |
| **One-Time** | 1 order AND review score > 2 (or no review) | Satisfied but not retained — re-engagement target |

In [21]:
avg_review_per_customer = (
    reviews
    .groupby("customer_unique_id")["review_score"]
    .mean()
    .reset_index()
    .rename(columns={"review_score": "avg_review_score"})
)

seg_df = customer_summary.merge(
    avg_review_per_customer,
    on="customer_unique_id",
    how="left",
)

revenue_p80 = seg_df["total_revenue"].quantile(0.80)

print(f"Revenue 80th-percentile threshold (High Value floor): R$ {revenue_p80:,.2f}")

Revenue 80th-percentile threshold (High Value floor): R$ 208.53


In [22]:
def assign_segment(row: pd.Series, revenue_threshold: float) -> str:
    """Assign a business segment based on purchase frequency and spend."""
    orders  = row["total_orders"]
    revenue = row["total_revenue"]
    review  = row["avg_review_score"]

    if orders >= 2 and revenue >= revenue_threshold:
        return "High Value"
    if orders >= 3:
        return "Loyal"
    if orders == 2:
        return "Occasional"
    if orders == 1 and pd.notna(review) and review <= 2:
        return "At Risk"
    return "One-Time"


seg_df["segment"] = seg_df.apply(
    assign_segment, axis=1, revenue_threshold=revenue_p80
)

segment_profile = (
    seg_df
    .groupby("segment")
    .agg(
        customers     = ("customer_unique_id", "count"),
        total_revenue = ("total_revenue",       "sum"),
        avg_revenue   = ("total_revenue",       "mean"),
        avg_orders    = ("total_orders",        "mean"),
        avg_review    = ("avg_review_score",    "mean"),
    )
    .reset_index()
)

total_c = segment_profile["customers"].sum()
total_r = segment_profile["total_revenue"].sum()

segment_profile["customer_pct"] = (segment_profile["customers"] / total_c * 100).round(1)
segment_profile["revenue_pct"]  = (segment_profile["total_revenue"] / total_r * 100).round(1)

segment_profile = segment_profile.sort_values("total_revenue", ascending=False)

print("Customer Segment Profiles:")
segment_profile[[
    "segment", "customers", "customer_pct",
    "total_revenue", "revenue_pct",
    "avg_revenue", "avg_orders", "avg_review",
]]

Customer Segment Profiles:


,segment,customers,customer_pct,total_revenue,revenue_pct,avg_revenue,avg_orders,avg_review
4,One-Time,78979,84.60,"12,366,039.33",80.20,156.57,1.00,4.58
0,At Risk,11578,12.40,"2,189,546.96",14.20,189.11,1.00,1.24
1,High Value,1535,1.60,"689,437.97",4.50,449.15,2.17,4.15
3,Occasional,1214,1.30,"166,840.93",1.10,137.43,2.00,4.25
2,Loyal,52,0.10,"7,908.56",0.10,152.09,3.10,4.59


In [23]:
SEGMENT_COLOURS = {
    "High Value": PALETTE["positive"],
    "Loyal":      PALETTE["primary"],
    "Occasional": PALETTE["neutral"],
    "At Risk":    PALETTE["negative"],
    "One-Time":   PALETTE["grey"],
}

fig_seg = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=("Share of Customers", "Share of Revenue"),
)

fig_seg.add_trace(
    go.Pie(
        labels=segment_profile["segment"],
        values=segment_profile["customers"],
        marker_colors=[SEGMENT_COLOURS[s] for s in segment_profile["segment"]],
        hole=0.45,
        textinfo="label+percent",
        name="Customers",
    ),
    row=1, col=1,
)

fig_seg.add_trace(
    go.Pie(
        labels=segment_profile["segment"],
        values=segment_profile["total_revenue"],
        marker_colors=[SEGMENT_COLOURS[s] for s in segment_profile["segment"]],
        hole=0.45,
        textinfo="label+percent",
        name="Revenue",
    ),
    row=1, col=2,
)

fig_seg.update_layout(
    title=dict(text="Customer Segmentation — Size vs Revenue Contribution", font=dict(size=16)),
    template=CHART_TEMPLATE,
    height=440,
    showlegend=True,
    legend=dict(orientation="v", x=1.02),
)

fig_seg.show()

In [24]:
fig_bubble = px.scatter(
    segment_profile,
    x="avg_orders",
    y="avg_revenue",
    size="customers",
    color="segment",
    color_discrete_map=SEGMENT_COLOURS,
    hover_name="segment",
    hover_data={"customers": True, "revenue_pct": True, "avg_review": True},
    title="Segment Landscape: Avg Orders vs Avg Revenue  (size = # Customers)",
    labels={
        "avg_orders":  "Avg Orders per Customer",
        "avg_revenue": "Avg Revenue per Customer (R$)",
    },
    template=CHART_TEMPLATE,
    size_max=75,
)

fig_bubble.update_layout(height=480)
fig_bubble.show()

In [25]:
high_value_geo = (
    seg_df[seg_df["segment"] == "High Value"]
    .groupby("state")["customer_unique_id"]
    .count()
    .reset_index(name="high_value_customers")
    .sort_values("high_value_customers", ascending=False)
    .head(10)
)

fig_hv_geo = px.bar(
    high_value_geo,
    x="state",
    y="high_value_customers",
    title="Top 10 States by High Value Customer Count",
    labels={"state": "State", "high_value_customers": "High Value Customers"},
    color="high_value_customers",
    color_continuous_scale="Greens",
    template=CHART_TEMPLATE,
)

fig_hv_geo.update_layout(height=380, coloraxis_showscale=False)
fig_hv_geo.show()

### Segmentation Insights

The segmentation reveals a classic marketplace long-tail dynamic:

- **One-Time customers dominate by headcount** but their individual revenue contribution is modest. The sheer scale of this segment means that even a 5% conversion to Occasional buyers is a significant revenue event without incremental acquisition spend.
- **High Value customers are disproportionate revenue contributors** — a small percentage of the customer base drives an outsized share of revenue. Any churn in this segment must be treated as a P1 incident by the Customer Success team.
- **The At Risk segment is a recoverable revenue pool** — these customers had a bad experience and did not return. A targeted service-recovery campaign could convert a meaningful portion back.
- **Loyal customers have the highest average review scores** — frequent buyers are the most satisfied, validating that the product works. The challenge is getting customers past their first order.

> **Strategic implication:** The highest-ROI lifecycle investment is converting One-Time buyers into Occasional buyers. The second purchase is the single most predictive indicator of long-term retention.

---
## 8 — Business Recommendations

The following recommendations are grounded directly in the analytical findings above.  
They are sized by business impact and scoped to a **90-day implementation horizon**.

### Priority 1 — Win the Second Purchase  *(Highest Impact)*

**Finding:** Most customers never return after their first order, despite reporting satisfactory review scores.  
**Action:** Launch a **post-purchase re-engagement sequence** triggered 14–21 days after first delivery:
- Day 14: product recommendations based on first purchase category
- Day 21: a 10–15% discount voucher with 30-day expiry
- Day 35: social proof email ('Customers like you also bought')

**Expected impact:** A 5% conversion of One-Time customers to a second purchase is a material GMV uplift at near-zero acquisition cost.

---

### Priority 2 — Protect High Value Customers

**Finding:** High Value customers are few in number but account for a disproportionate share of revenue.  
**Action:** Introduce an **Olist VIP Programme** with priority support SLA, free expedited shipping, and early access to promotions. Review eligibility quarterly based on rolling 12-month spend.

**Expected impact:** A 10% reduction in High Value customer churn preserves significant annual revenue.

---

### Priority 3 — Fix Delivery to Fix Satisfaction

**Finding:** Delivery delay is the single strongest predictor of low review scores.  
**Action:** Implement **Proactive Delay Communication**: automatic apology + R$ 15 store credit when an order tracks 2+ days late. Set seller dispatch SLAs with performance warnings for >15% late-pickup rate.

**Expected impact:** Moving average review score toward 4.5 increases repeat purchase rate and NPS.

---

### Priority 4 — At Risk Customer Recovery

**Finding:** At Risk customers are identifiable — one order, score of 1 or 2.  
**Action:** Deploy a **Service Recovery Campaign** within 48 hours of a low review: named apology, R$ 20–30 store credit, and offer of free return.

**Expected impact:** Industry benchmarks suggest 10–20% of service-recovered customers make a subsequent purchase.

---

### Priority 5 — Regional Expansion

**Finding:** SP, RJ, and MG dominate. Sul and Nordeste are underrepresented relative to their population.  
**Action:** Geo-targeted acquisition campaigns in Porto Alegre, Curitiba, Recife, and Fortaleza, combined with local seller recruitment to reduce delivery times in those regions.

---

### 90-Day Implementation Roadmap

| Timeframe | Initiative | Owner | Primary KPI |
|---|---|---|---|
| 0–30 days | Service recovery for At Risk segment | Customer Success | Recovery rate |
| 0–30 days | VIP programme for High Value segment | Marketing | HV churn rate |
| 30–60 days | Post-purchase re-engagement sequence | CRM / Marketing | 2nd purchase rate |
| 30–60 days | Proactive delay communication protocol | Operations | Avg review (late orders) |
| 60–90 days | Geo-targeted campaigns (Sul + Nordeste) | Growth | New customers in target states |
| Ongoing | Monthly segment migration dashboard | Analytics | Segment transition rates |

---
## 9 — Appendix: Data Notes

| Item | Detail |
|---|---|
| Source dataset | `analytics_base_table.parquet` (generated by Notebook 03) |
| Analysis scope | Delivered orders only |
| Revenue metric | `total_order_value` = `price` + `freight_value` at order-item grain |
| Customer identity | `customer_unique_id` (de-duplicated across all `customer_id` instances) |
| Review grain | Deduplicated to one review per `order_id` |
| Cohort definition | Month of first delivered order |
| Segmentation | Rule-based; revenue threshold = 80th percentile of customer total revenue |
| Date coverage | January 2017 – August 2018 (partial months excluded from trend charts) |

---
*Notebook authored as part of the Olist Brazilian E-Commerce Analytics project — Gyakriti.*